# 基于 YOLOv8 的实时口罩佩戴检测系统

> **课程**: 数据分析与数据挖掘 — 端到端目标检测系统开发  
> **数据集**: [Face Mask Detection (Kaggle)](https://www.kaggle.com/datasets/andrewmvd/face-mask-detection), 853 张图像, 3 类别  
> **GPU**: NVIDIA RTX 4090 / 24GB（实验环境参考）

## 三类检测目标

| ID | 类别 | 含义 |
|:--:|:---|:---|
| 0 | `with_mask` | 正确佩戴口罩 |
| 1 | `without_mask` | 未佩戴口罩 |
| 2 | `mask_weared_incorrect` | 不规范佩戴 |

## 实验流程总览

```
数据下载 → VOC→YOLO 格式转换 → EDA 分布分析 → 离线增强(过采样)
  → Phase 0: 原始数据 Baseline
  → Phase 2: 增强策略消融 (BestAug = Mosaic+MixUp)
  → Phase 3: 模型尺度对比 (YOLOv8m)
  → 模型评估 (mAP + FPS) + 推理可视化
```

完整 12 组消融实验参见项目仓库 `run_ablation.py` 与 `experiment_results.md`。

## 0. 环境准备与工作区初始化

In [ ]:
# ── 安装依赖 ──────────────────────────────────
!pip install -q torch>=2.0.0 torchvision>=0.15.0 ultralytics>=8.0.0
!pip install -q opencv-python>=4.5.0 matplotlib>=3.5.0 seaborn>=0.12.0
!pip install -q tensorboard>=2.12.0 tqdm>=4.64.0 pyyaml>=6.0 kagglehub>=0.1.0

In [ ]:
# ── 工作区目录 ─────────────────────────────────
from pathlib import Path
import os

WORKSPACE = Path.cwd() / "workspace"
DATA_RAW = WORKSPACE / "data" / "raw"
DATA_PROCESSED = WORKSPACE / "data" / "processed"
DATA_AUGMENTED = WORKSPACE / "data" / "augmented"
CONFIGS = WORKSPACE / "configs"
EXPERIMENTS = WORKSPACE / "experiments"
REPORTS = WORKSPACE / "reports" / "figures"
VIS_OUT = WORKSPACE / "visualization_results"

for d in [DATA_RAW, DATA_PROCESSED, DATA_AUGMENTED, CONFIGS, EXPERIMENTS, REPORTS, VIS_OUT]:
    d.mkdir(parents=True, exist_ok=True)

print(f"工作区: {WORKSPACE.resolve()}")
for d in sorted(WORKSPACE.rglob("*")):
    if d.is_dir() and not any(p.startswith('__') for p in d.parts):
        print(f"  {d.relative_to(WORKSPACE)}/")

## 1. 数据下载

从 Kaggle 下载 `andrewmvd/face-mask-detection` 数据集并解压到 `workspace/data/raw/`。

In [ ]:
import zipfile
import kagglehub

print(f"下载 andrewmvd/face-mask-detection → {DATA_RAW}")
archive_path = kagglehub.dataset_download(
    "andrewmvd/face-mask-detection", path=DATA_RAW, force_download=True
)
archive_path = Path(archive_path)

# 解压 zip
zip_files = list(archive_path.rglob("*.zip"))
if zip_files:
    print(f"解压 {len(zip_files)} 个压缩包...")
    for zf in zip_files:
        with zipfile.ZipFile(zf, "r") as z:
            z.extractall(DATA_RAW)
        zf.unlink()
    print("解压完成。")

# 清理空目录
for subdir in sorted(DATA_RAW.rglob("*"), reverse=True):
    if subdir.is_dir() and not any(subdir.iterdir()):
        subdir.rmdir()

print("\n数据文件：")
for item in sorted(DATA_RAW.iterdir()):
    files = len(list(item.rglob("*"))) if item.is_dir() else 1
    print(f"  {item.name}/ ({files} 项)" if item.is_dir() else f"  {item.name}")

## 2. VOC XML → YOLO TXT 格式转换 + 分层划分

将 PASCAL VOC 标注转为 YOLOv8 所需的归一化 txt 格式，按 8:1:1 分层划分为 train/val/test。

**分层策略**：每张图片按其出现最多的类别分组，组内按比例随机划分，确保三个集合的类别分布一致（seed=42）。

In [ ]:
import glob
import math
import random
import shutil
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict

CLASS_MAP = {"with_mask": 0, "without_mask": 1, "mask_weared_incorrect": 2}
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

def parse_xml(xml_path: str) -> tuple:
    """解析 PASCAL VOC XML，返回 (文件名, [(cls_id, xc, yc, bw, bh), ...])。
    坐标归一化到 [0,1]，跳过 difficult=1 的目标。"""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    filename = root.find("filename").text
    size = root.find("size")
    width = float(size.find("width").text)
    height = float(size.find("height").text)

    boxes = []
    for obj in root.findall("object"):
        difficult = obj.find("difficult")
        if difficult is not None and int(difficult.text) == 1:
            continue
        name = obj.find("name").text
        cls_id = CLASS_MAP[name]
        bndbox = obj.find("bndbox")
        xmin = float(bndbox.find("xmin").text)
        ymin = float(bndbox.find("ymin").text)
        xmax = float(bndbox.find("xmax").text)
        ymax = float(bndbox.find("ymax").text)
        xc = (xmin + xmax) / 2.0 / width
        yc = (ymin + ymax) / 2.0 / height
        bw = (xmax - xmin) / width
        bh = (ymax - ymin) / height
        boxes.append((cls_id, xc, yc, bw, bh))
    return filename, boxes


def stratified_split(records: list, ratios=(0.80, 0.10, 0.10)) -> list:
    """按主类别分层划分。使用 math.floor 避免 banker's rounding 导致小组 val/test 为空。"""
    groups = defaultdict(list)
    for rec in records:
        boxes = rec[1]
        if not boxes:
            continue
        class_counts = Counter(b[0] for b in boxes)
        primary_class = class_counts.most_common(1)[0][0]
        groups[primary_class].append(rec)

    splits = [[], [], []]
    for group in groups.values():
        random.shuffle(group)
        n = len(group)
        n_train = math.floor(n * ratios[0])
        n_val = math.floor(n * ratios[1])
        # 确保验证集至少分配 1 张图片（当组内样本 ≥ 5 时）
        if n_val == 0 and n >= 5:
            n_val = 1
        splits[0].extend(group[:n_train])
        splits[1].extend(group[n_train:n_train + n_val])
        splits[2].extend(group[n_train + n_val:])
    return splits


def write_yolo_label(boxes: list, txt_path: Path) -> None:
    """写入 YOLO TXT 标注（6 位小数精度）。"""
    with open(txt_path, "w", encoding="utf-8") as f:
        for cls_id, xc, yc, bw, bh in boxes:
            f.write(f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")


# ── 执行转换 ────────────────────────────────────
src_images = DATA_RAW / "images"
src_annotations = DATA_RAW / "annotations"
dst_images = DATA_PROCESSED / "images"
dst_labels = DATA_PROCESSED / "labels"

xml_paths = sorted(glob.glob(str(src_annotations / "*.xml")))
print(f"找到 {len(xml_paths)} 个标注文件。")

records = []
parse_errors = 0
for xml_path in xml_paths:
    try:
        records.append(parse_xml(xml_path))
    except Exception as e:
        print(f"  跳过 {Path(xml_path).name}: {e}")
        parse_errors += 1
print(f"成功解析 {len(records)} 个标注 ({parse_errors} 个错误)。")

splits = stratified_split(records)
split_names = ["train", "val", "test"]

for name in split_names:
    (dst_images / name).mkdir(parents=True, exist_ok=True)
    (dst_labels / name).mkdir(parents=True, exist_ok=True)

src_images_map = {p.stem: p for p in Path(src_images).iterdir() if p.is_file()}
split_stats = {}
all_class_counts = Counter()

for name, split_records in zip(split_names, splits):
    class_counts = Counter()
    for filename, boxes in split_records:
        stem = Path(filename).stem
        txt_path = dst_labels / name / f"{stem}.txt"
        write_yolo_label(boxes, txt_path)

        ext = None
        for candidate in src_images_map.values():
            if candidate.stem == stem:
                ext = candidate.suffix
                break
        if ext is None:
            print(f"  Warning: 未找到图片 {filename}")
            continue
        shutil.copy2(src_images / f"{stem}{ext}", dst_images / name / f"{stem}{ext}")
        class_counts.update(b[0] for b in boxes)
    split_stats[name] = {"images": len(split_records), "instances": class_counts}
    all_class_counts.update(class_counts)

# 打印摘要
cls_names = {v: k for k, v in CLASS_MAP.items()}
print("\n" + "=" * 55)
print("数据集转换摘要")
print("=" * 55)
total_images = sum(s["images"] for s in split_stats.values())
print(f"总图片数:          {total_images}")
for name in split_names:
    s = split_stats[name]
    print(f"  {name:6s}:        {s['images']:4d} 张图片  "
          f"{sum(s['instances'].values()):4d} 个实例")
print(f"\n总实例数:          {sum(all_class_counts.values())}")
print("各类别分布:")
for cls_id, count in sorted(all_class_counts.items()):
    print(f"  {cls_names[cls_id]:30s} (id={cls_id}): {count:5d}")
print("=" * 55)

## 3. 类别分布 EDA

分析数据集的长尾不平衡分布，输出柱状图至 `workspace/reports/figures/`。

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# 统计全部划分的类别实例数
all_counts = Counter()
for split in ["train", "val", "test"]:
    label_dir = DATA_PROCESSED / "labels" / split
    if label_dir.exists():
        for txt_file in label_dir.glob("*.txt"):
            with open(txt_file, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        all_counts[int(parts[0])] += 1

class_names = {0: "with_mask", 1: "without_mask", 2: "mask_weared_incorrect"}
categories = [class_names[i] for i in sorted(all_counts.keys())]
counts = [all_counts[i] for i in sorted(all_counts.keys())]

# 绘制
plt.figure(figsize=(8, 5))
colors = ["#2ecc71", "#e74c3c", "#f39c12"]
bars = plt.bar(categories, counts, color=colors)
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
             f"{count}", ha="center", va="bottom", fontsize=11)
plt.title("Long-Tail Distribution of Mask Wearing Classes", fontsize=14)
plt.ylabel("Number of Instances", fontsize=12)
plt.xlabel("Class", fontsize=12)

ratio = max(counts) / min(counts)
plt.figtext(0.15, 0.85, f"Imbalance ratio: {ratio:.1f}:1",
            bbox=dict(facecolor="white", alpha=0.8))

plt.savefig(REPORTS / "longtail_distribution.png", dpi=300, bbox_inches="tight")
print(f"图表已保存至: {REPORTS / 'longtail_distribution.png'}")
print(f"不平衡比例 (最大/最小): {ratio:.2f}")
plt.show()

## 4. 离线数据增强

对少数类（类别 1 `without_mask`、类别 2 `mask_weared_incorrect`）进行**离线过采样**，使其实例数达到类别 0 的 60%。

增强策略：
- 几何变换：随机水平翻转、旋转（±15°）、缩放（0.9~1.1）、平移（±5%）
- 光度变换：亮度（±30）、对比度（0.8~1.2）、高斯噪声（σ≤10）

增强后不平衡比从 **26:1 降至约 1.7:1**。

In [ ]:
import random
import shutil
from collections import Counter

import cv2
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

import yaml

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def augment_image_and_boxes(image, boxes, transform_params):
    """对图像和边界框同步施加数据增强。"""
    h, w = image.shape[:2]

    # 1. 水平翻转
    if transform_params['flip']:
        image = cv2.flip(image, 1)
        boxes = [[cls_id, 1.0 - xc, yc, bw, bh] for cls_id, xc, yc, bw, bh in boxes]

    # 2. 亮度 & 对比度
    image = cv2.convertScaleAbs(image, alpha=transform_params['contrast'],
                                beta=transform_params['brightness'])

    # 3. 仿射变换（旋转 + 缩放 + 平移）
    center = (w / 2, h / 2)
    rot_mat = cv2.getRotationMatrix2D(center, transform_params['angle'],
                                       transform_params['scale'])
    rot_mat[0, 2] += transform_params['tx'] * w
    rot_mat[1, 2] += transform_params['ty'] * h
    image = cv2.warpAffine(image, rot_mat, (w, h), flags=cv2.INTER_LINEAR,
                           borderMode=cv2.BORDER_REPLICATE)

    # 变换边界框
    augmented_boxes = []
    for cls_id, xc, yc, bw, bh in boxes:
        x1, y1 = (xc - bw / 2) * w, (yc - bh / 2) * h
        x2, y2 = (xc + bw / 2) * w, (yc + bh / 2) * h
        corners = np.array([[x1, y1, 1], [x2, y1, 1], [x1, y2, 1], [x2, y2, 1]])
        transformed = corners @ rot_mat.T
        nx1, nx2 = np.clip([np.min(transformed[:, 0]), np.max(transformed[:, 0])], 0, w - 1)
        ny1, ny2 = np.clip([np.min(transformed[:, 1]), np.max(transformed[:, 1])], 0, h - 1)
        nbw, nbh = (nx2 - nx1) / w, (ny2 - ny1) / h
        if nbw > 0.001 and nbh > 0.001:
            augmented_boxes.append([cls_id, (nx1 + nx2) / (2 * w), (ny1 + ny2) / (2 * h), nbw, nbh])

    # 4. 高斯噪声
    if transform_params['noise'] > 0:
        noise = np.random.normal(0, transform_params['noise'], image.shape).astype(np.int16)
        image = np.clip(image.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    return image, augmented_boxes


# ── 执行增强 ────────────────────────────────────
TARGET_RATIO = 0.6
train_img_dir = DATA_PROCESSED / "images" / "train"
train_lbl_dir = DATA_PROCESSED / "labels" / "train"

# 统计各类别实例数
class_counts = Counter()
image_to_classes = {}
for lbl_file in train_lbl_dir.glob("*.txt"):
    with open(lbl_file, 'r') as f:
        classes = [int(line.split()[0]) for line in f.readlines() if line.strip()]
        class_counts.update(classes)
        image_to_classes[lbl_file.stem] = classes

print(f"原始训练集实例: {dict(class_counts)}")
target_count = int(class_counts[0] * TARGET_RATIO)
print(f"少数类目标实例数: {target_count} (类别0的 {TARGET_RATIO*100:.0f}%)")

# 创建输出目录 & 复制 val 和 test（test 原样复制，不参与增强）
for split in ["train", "val", "test"]:
    (DATA_AUGMENTED / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATA_AUGMENTED / "labels" / split).mkdir(parents=True, exist_ok=True)

print("复制验证集和测试集...")
for split in ["val", "test"]:
    src_img_split = DATA_PROCESSED / "images" / split
    src_lbl_split = DATA_PROCESSED / "labels" / split
    if src_img_split.exists():
        for f in src_img_split.glob("*.*"):
            shutil.copy(f, DATA_AUGMENTED / "images" / split / f.name)
    if src_lbl_split.exists():
        for f in src_lbl_split.glob("*.txt"):
            shutil.copy(f, DATA_AUGMENTED / "labels" / split / f.name)

print("复制原始训练数据...")
for f in train_img_dir.glob("*.*"):
    shutil.copy(f, DATA_AUGMENTED / "images" / "train" / f.name)
for f in train_lbl_dir.glob("*.txt"):
    shutil.copy(f, DATA_AUGMENTED / "labels" / "train" / f.name)

# 对类别 1, 2 进行过采样增强
for cls_id in [1, 2]:
    needed = target_count - class_counts[cls_id]
    if needed <= 0:
        print(f"类别 {cls_id} 实例已足够 ({class_counts[cls_id]})，跳过。")
        continue

    print(f"增强类别 {cls_id}，需生成约 {needed} 个额外实例 ...")
    candidate_stems = [s for s, cs in image_to_classes.items() if cls_id in cs]
    if not candidate_stems:
        print(f"Warning: 类别 {cls_id} 无样本")
        continue

    generated = 0
    pbar = tqdm(total=needed)
    while generated < needed:
        stem = random.choice(candidate_stems)
        img_path = None
        for ext in ['.jpg', '.png', '.jpeg']:
            if (train_img_dir / f"{stem}{ext}").exists():
                img_path = train_img_dir / f"{stem}{ext}"
                break
        if not img_path:
            continue

        try:
            pil_img = Image.open(str(img_path)).convert("RGB")
            image = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        except Exception:
            continue

        with open(train_lbl_dir / f"{stem}.txt", 'r') as f:
            boxes = [list(map(float, line.split())) for line in f.readlines() if line.strip()]
            for b in boxes:
                b[0] = int(b[0])

        params = {
            'flip': random.random() > 0.5,
            'angle': random.uniform(-15, 15),
            'scale': random.uniform(0.9, 1.1),
            'tx': random.uniform(-0.05, 0.05),
            'ty': random.uniform(-0.05, 0.05),
            'brightness': random.uniform(-30, 30),
            'contrast': random.uniform(0.8, 1.2),
            'noise': random.uniform(0, 10) if random.random() > 0.5 else 0,
        }
        aug_img, aug_boxes = augment_image_and_boxes(image, boxes, params)
        if aug_boxes:
            new_stem = f"{stem}_aug_{cls_id}_{generated}"
            cv2.imwrite(str(DATA_AUGMENTED / "images" / "train" / f"{new_stem}.jpg"), aug_img)
            with open(DATA_AUGMENTED / "labels" / "train" / f"{new_stem}.txt", 'w') as f:
                for b in aug_boxes:
                    f.write(f"{int(b[0])} {' '.join(f'{x:.6f}' for x in b[1:])}\n")
            added = sum(1 for b in aug_boxes if b[0] == cls_id)
            generated += added
            pbar.update(added)
    pbar.close()

print("\n增强完成。")

# 统计增强后分布
aug_counts = Counter()
for lbl_file in (DATA_AUGMENTED / "labels" / "train").glob("*.txt"):
    with open(lbl_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                aug_counts[int(parts[0])] += 1
print(f"增强后训练集实例: {dict(aug_counts)}")
print(f"不平衡比: {max(aug_counts.values()) / min(aug_counts.values()):.1f}:1")

## 5. 生成数据集配置文件

动态生成两个 YAML 配置：原始数据集（用于 Baseline）和均衡数据集（用于消融/尺度对比）。

In [ ]:
# 原始数据集配置（含 train/val/test 三个划分）
dataset_yaml = CONFIGS / "dataset.yaml"
dataset_yaml.write_text(f"""\
path: {DATA_PROCESSED.resolve().as_posix()}
train: images/train
val: images/val
test: images/test
nc: 3
names:
  0: with_mask
  1: without_mask
  2: mask_weared_incorrect
""", encoding="utf-8")

# 均衡数据集配置（增强后，含 train/val/test 三个划分）
balanced_yaml = CONFIGS / "dataset_balanced.yaml"
balanced_yaml.write_text(f"""\
path: {DATA_AUGMENTED.resolve().as_posix()}
train: images/train
val: images/val
test: images/test
nc: 3
names:
  0: with_mask
  1: without_mask
  2: mask_weared_incorrect
""", encoding="utf-8")

print(f"✓ {dataset_yaml}")
print(f"✓ {balanced_yaml}")

## 6. 阶段 0 — 原始数据基线

在**原始不平衡数据**上，**关闭所有数据增强**，训练 YOLOv8n 作为相对对照。

> ⏱ 预计耗时: ~15 分钟 (RTX 4090)

In [ ]:
from ultralytics import YOLO

baseline_dir = EXPERIMENTS / "baseline"
best_pt = baseline_dir / "weights" / "best.pt"

if best_pt.exists():
    print(f"[SKIP] Phase 0 Baseline 已完成 → {baseline_dir}")
else:
    print("=" * 60)
    print("  Phase 0: Baseline（原始不平衡数据，关闭所有增强）")
    print("=" * 60)

    model = YOLO("yolov8n.pt")
    model.train(
        data=str(dataset_yaml),
        epochs=100,
        batch=16,
        imgsz=640,
        patience=50,
        mosaic=0.0,
        mixup=0.0,
        project=str(EXPERIMENTS),
        name="baseline",
        exist_ok=True,
    )

## 7. 阶段 2 — 增强策略消融（BestAug）

在**均衡数据集**上，使用 **Mosaic=1.0 + MixUp=0.5**（消融实验验证的最佳组合），训练 YOLOv8n。

> ⏱ 预计耗时: ~20 分钟 (RTX 4090)  
> 完整 8 组对照矩阵参见 `experiment_results.md`。

In [ ]:
ablation_dir = EXPERIMENTS / "ablation" / "abl5_mosaic_mixup"
best_pt_abl = ablation_dir / "weights" / "best.pt"

if best_pt_abl.exists():
    print(f"[SKIP] Phase 2 BestAug 已完成 → {ablation_dir}")
else:
    print("=" * 60)
    print("  Phase 2: BestAug — Mosaic=1.0, MixUp=0.5（均衡数据）")
    print("=" * 60)

    model = YOLO("yolov8n.pt")
    model.train(
        data=str(balanced_yaml),
        epochs=100,
        batch=16,
        imgsz=640,
        patience=50,
        mosaic=1.0,
        mixup=0.5,
        close_mosaic=0,
        project=str(EXPERIMENTS / "ablation"),
        name="abl5_mosaic_mixup",
        exist_ok=True,
    )

## 8. 阶段 3 — 模型尺度对比（BestModel）

在 BestAug（Mosaic=1.0, MixUp=0.5）下训练 **YOLOv8m**（精度最优）。

> ⏱ 预计耗时: ~40 分钟 (RTX 4090)  
> 完整 n/s/m 三维对比参见 `experiment_results.md`。

In [ ]:
scale_dir = EXPERIMENTS / "scale" / "scale_m"
best_pt_scale = scale_dir / "weights" / "best.pt"

if best_pt_scale.exists():
    print(f"[SKIP] Phase 3 BestModel 已完成 → {scale_dir}")
else:
    print("=" * 60)
    print("  Phase 3: YOLOv8m + BestAug（Mosaic=1.0, MixUp=0.5）")
    print("=" * 60)

    model = YOLO("yolov8m.pt")
    model.train(
        data=str(balanced_yaml),
        epochs=100,
        batch=16,
        imgsz=640,
        patience=50,
        mosaic=1.0,
        mixup=0.5,
        close_mosaic=0,
        project=str(EXPERIMENTS / "scale"),
        name="scale_m",
        exist_ok=True,
    )

## 9. 模型评估

在**留出的 test 划分**上评估最佳模型的 mAP、Precision、Recall 及推理速度（FPS）。

> 优先使用增强数据的 test 划分；若增强未运行，自动回退到原始数据的 test 划分。

In [ ]:
import json
import time

from ultralytics import YOLO

# 自动选择已完成的最佳权重
def find_best_weights():
    candidates = [
        EXPERIMENTS / "scale" / "scale_m" / "weights" / "best.pt",
        EXPERIMENTS / "ablation" / "abl5_mosaic_mixup" / "weights" / "best.pt",
        EXPERIMENTS / "baseline" / "weights" / "best.pt",
    ]
    for c in candidates:
        if c.exists():
            return c
    return None


MODEL_PATH = find_best_weights()
if MODEL_PATH is None:
    print("⚠ 未找到训练好的权重文件，请先运行至少一个训练阶段。")
else:
    print(f"评估模型: {MODEL_PATH}")

    # 选择数据配置：优先使用增强数据，若未运行增强则回退到原始数据 test 划分
    if (DATA_AUGMENTED / "images" / "test").exists():
        eval_data = str(balanced_yaml)
        eval_split = "test"
        print(f"使用均衡数据集 (test 划分)")
    elif (DATA_PROCESSED / "images" / "test").exists():
        eval_data = str(dataset_yaml)
        eval_split = "test"
        print(f"⚠ 增强数据不可用，回退到原始数据集 (test 划分)")
    else:
        eval_data = str(dataset_yaml)
        eval_split = "val"
        print(f"⚠ 使用 val 划分（test 划分不可用）")

    model = YOLO(str(MODEL_PATH))
    start = time.time()
    results = model.val(data=eval_data, split=eval_split,
                        project=str(EXPERIMENTS), name="evaluation", exist_ok=True)

    mAP50 = results.box.map50
    mAP95 = results.box.map
    precision = results.box.mp
    recall = results.box.mr

    speed = results.speed
    total_ms = speed.get("preprocess", 0) + speed.get("inference", 0) + speed.get("postprocess", 0)
    fps = 1000 / total_ms if total_ms > 0 else 0

    print("\n" + "=" * 45)
    print(f"  评估结果: {MODEL_PATH.name}")
    print(f"  mAP@0.5:       {mAP50:.4f}")
    print(f"  mAP@0.5:0.95:  {mAP95:.4f}")
    print(f"  Precision:     {precision:.4f}")
    print(f"  Recall:        {recall:.4f}")
    print(f"  推理速度:      {total_ms:.1f} ms/img  (FPS: {fps:.1f})")
    print(f"  评估划分:      {eval_split}")
    print("=" * 45)

    report = {
        "model": str(MODEL_PATH),
        "eval_split": eval_split,
        "mAP50": float(mAP50),
        "mAP95": float(mAP95),
        "precision": float(precision),
        "recall": float(recall),
        "fps": float(fps),
        "ms_per_img": float(total_ms),
    }
    report_path = EXPERIMENTS / "evaluation" / "metrics_report.json"
    report_path.parent.mkdir(parents=True, exist_ok=True)
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=4, ensure_ascii=False)
    print(f"指标报告已保存: {report_path}")

## 10. 推理可视化

对 `workspace/test_images/` 下的所有图片运行检测，结果保存至 `workspace/visualization_results/`。

（若无需可视化可跳过本节。）

In [ ]:
from ultralytics import YOLO

# 提示：将待检测图片放入 workspace/test_images/ 目录
TEST_IMAGES = WORKSPACE / "test_images"
TEST_IMAGES.mkdir(exist_ok=True)

image_files = (list(TEST_IMAGES.glob("*.jpg")) +
               list(TEST_IMAGES.glob("*.png")) +
               list(TEST_IMAGES.glob("*.jpeg")))

# 安全获取 MODEL_PATH（可能未运行第 9 节）
try:
    _model_available = MODEL_PATH is not None
except NameError:
    _model_available = False

if not image_files:
    print(f"将待检测图片放入 {TEST_IMAGES.resolve()} 后重新运行此 Cell。")
elif not _model_available:
    print("请先运行第 9 节（模型评估 Cell）以加载模型权重。")
else:
    model = YOLO(str(MODEL_PATH))
    VIS_OUT.mkdir(exist_ok=True)
    for img_path in image_files:
        print(f"处理: {img_path.name}")
        results = model(str(img_path))
        results[0].save(str(VIS_OUT / img_path.name))
    print(f"\n检测结果已保存至: {VIS_OUT.resolve()}")

## 11. 实验结论

| 指标 | 值 |
|:---|---:|
| 最佳增强策略 | Mosaic=1.0 + MixUp=0.5（协同比 Baseline 提升 6.9% mAP@50）|
| 最佳模型 | YOLOv8m（mAP@50 = 0.8608）|
| 性价比首选 | YOLOv8n（仅低 0.64% mAP@50，参数少 8 倍）|
| Copy-Paste | 无效（4 组对照实验指标完全一致）|

详细实验数据与完整分析见项目仓库 `experiment_results.md`。